In [2]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nltk.download('stopwords')
nltk.download('wordnet')

# ==============================
# LOAD JSON DATASET
# ==============================
df = pd.read_json(r"C:\Users\srikala\OneDrive\Desktop\NLP\Cell_Phones_and_Accessories_5.json", lines=True)

# ==============================
# SELECT REQUIRED COLUMNS
# ==============================
df = df[['reviewText', 'overall']]

# ==============================
# CONVERT RATING TO SENTIMENT
# ==============================
def convert_label(rating):
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

df['label'] = df['overall'].apply(convert_label)
df['text'] = df['reviewText']

# ==============================
# CLEAN DATASET
# ==============================
df = df[['text', 'label']]
df = df.dropna()

print("Dataset Shape:", df.shape)
print("\nLabel Distribution:\n", df['label'].value_counts())

# ==============================
# NLP PREPROCESSING
# ==============================
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]
    
    return " ".join(words)

# Apply preprocessing
df['clean_text'] = df['text'].apply(clean_text)

print("\nSample Cleaned Data:")
print(df[['text','clean_text']].head())

# ==============================
# FEATURE ENGINEERING (TF-IDF)
# ==============================
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_text'])
y = df['label']

# ==============================
# TRAIN-TEST SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==============================
# MODEL TRAINING
# ==============================

# Logistic Regression
lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)
pred_nb = nb.predict(X_test)

# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
pred_dt = dt.predict(X_test)

# ==============================
# EVALUATION FUNCTION
# ==============================
def evaluate(name, y_test, pred):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred, average='weighted'))
    print("Recall:", recall_score(y_test, pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, pred, average='weighted'))

# ==============================
# EVALUATE MODELS
# ==============================
evaluate("Logistic Regression", y_test, pred_lr)
evaluate("Naive Bayes", y_test, pred_nb)
evaluate("Decision Tree", y_test, pred_dt)

# ==============================
#SAMPLE PREDICTION
# ==============================
sample = ["This product is amazing and works perfectly"]
sample_clean = [clean_text(s) for s in sample]
sample_vec = vectorizer.transform(sample_clean)

print("\nSample Prediction:", lr.predict(sample_vec))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\srikala\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\srikala\AppData\Roaming\nltk_data...


Dataset Shape: (194439, 2)

Label Distribution:
 label
positive    148657
negative     24343
neutral      21439
Name: count, dtype: int64

Sample Cleaned Data:
                                                text  \
0  They look good and stick good! I just don't li...   
1  These stickers work like the review says they ...   
2  These are awesome and make my phone look so st...   
3  Item arrived in great time and was in perfect ...   
4  awesome! stays on, and looks great. can be use...   

                                          clean_text  
0  look good stick good like rounded shape always...  
1  sticker work like review say stick great stay ...  
2  awesome make phone look stylish used one far a...  
3  item arrived great time perfect condition howe...  
4  awesome stay look great used multiple apple pr...  

Logistic Regression
Accuracy: 0.827504628677227
Precision: 0.7949618784047104
Recall: 0.827504628677227
F1 Score: 0.8003150351347709

Naive Bayes
Accuracy: 0.76838613454021